In [93]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [94]:
print("meow")

meow


# MY CODE


In [95]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import wandb
from kaggle_secrets import UserSecretsClient
import warnings
warnings.filterwarnings('ignore')

options = ['A', 'B', 'C', 'D', 'E']

secrets = UserSecretsClient()
wandb.login(key=secrets.get_secret("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [96]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

print("Train shape:", train.shape)
print("Test shape :", test.shape)

Train shape: (2000, 8)
Test shape : (500, 7)


In [97]:
def combine_prompt_option(row, option):
    return row['prompt'] + " " + row[option]

def map_at_3(df, predict_fn):
    scores = []
    for _, row in df.iterrows():
        prediction = predict_fn(row)
        predicted_labels = prediction.split()
        correct = row['answer']
        score = 0.0
        if correct in predicted_labels:
            rank = predicted_labels.index(correct) + 1
            score = 1.0 / rank
        scores.append(score)
    return np.mean(scores)

In [98]:
class LogisticRegressionScratch:
    def __init__(self, lr=0.1, epochs=200, l2=0.01):
        self.lr = lr
        self.epochs = epochs
        self.l2 = l2
        self.weights = None
        self.bias = None

    def sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0

        for epoch in range(self.epochs):
            linear_output = np.dot(X, self.weights) + self.bias
            y_pred = self.sigmoid(linear_output)

            dw = (1 / n_samples) * np.dot(X.T, (y_pred - y)) + self.l2 * self.weights
            db = (1 / n_samples) * np.sum(y_pred - y)

            self.weights -= self.lr * dw
            self.bias -= self.lr * db

            if epoch % 100 == 0:
                loss = -np.mean(y * np.log(y_pred + 1e-9) + (1 - y) * np.log(1 - y_pred + 1e-9))
                print(f"Epoch {epoch}: loss={loss:.4f}")

    def predict_proba(self, X):
        linear_output = np.dot(X, self.weights) + self.bias
        return self.sigmoid(linear_output)

In [99]:
train_split, val_split = train_test_split(train, test_size=0.2, random_state=42)

texts_train, labels_train = [], []
for _, row in train_split.iterrows():
    for opt in options:
        texts_train.append(combine_prompt_option(row, opt))
        labels_train.append(1 if opt == row['answer'] else 0)

print(f"Total training pairs: {len(texts_train)}")
print(f"Positive pairs: {sum(labels_train)}")

Total training pairs: 8000
Positive pairs: 1600


In [100]:
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2), min_df=2, max_df=0.9)
X_train = vectorizer.fit_transform(texts_train).toarray()
y_train = np.array(labels_train)

print("X_train shape:", X_train.shape)

clf = LogisticRegressionScratch(lr=0.21, epochs=1200, l2=0.01)
clf.fit(X_train, y_train)

X_train shape: (8000, 10000)
Epoch 0: loss=0.6931
Epoch 100: loss=0.4995
Epoch 200: loss=0.4986
Epoch 300: loss=0.4981
Epoch 400: loss=0.4976
Epoch 500: loss=0.4972
Epoch 600: loss=0.4969
Epoch 700: loss=0.4966
Epoch 800: loss=0.4964
Epoch 900: loss=0.4963
Epoch 1000: loss=0.4961
Epoch 1100: loss=0.4960


In [101]:
def predict_top3_logreg(row):
    texts_row = [combine_prompt_option(row, opt) for opt in options]
    X_row = vectorizer.transform(texts_row).toarray()
    scores = clf.predict_proba(X_row)
    top3_indices = scores.argsort()[::-1][:3]
    return ' '.join([options[i] for i in top3_indices])

In [102]:
score = map_at_3(val_split, predict_top3_logreg)
print(f"TF-IDF + LogReg (scratch) Val MAP@3: {score:.4f}")

run = wandb.init(
    entity="varnitchourasiya27-indian-institute-of-technology-madras",
    project="23f3000843-t22026",
    name="tfidf-logreg-scratch",
    config={"model": "tfidf_logreg_scratch", "max_features": 10000, "ngram_range": "(1,2)", "epochs": 1200, "lr": 0.2, "l2": 0.01}
)
wandb.log({"MAP@3": score})
wandb.finish()

TF-IDF + LogReg (scratch) Val MAP@3: 0.9450


MAP@3,▁
MAP@3,0.945


In [103]:
test['Prediction'] = test.apply(predict_top3_logreg, axis=1)
submission = test[['id', 'Prediction']].copy()
submission.columns = ['ID', 'Prediction']
submission.to_csv('submission.csv', index=False)
print(submission.head())

   ID Prediction
0   1      A E C
1   2      B D C
2   3      B E D
3   4      E C A
4   5      C A B
